# tpt-engineering: Python tour

The `tpt_eng` package wraps the Rust crates. Units are in the names:
`_k` kelvin, `_pa` pascal, `_kj_per_kg`, `_nm` newton-metre, `_s` seconds.

Install once with `pip install ./crates/tpt-eng-py` from the repository root.

## Water and steam (IAPWS-IF97)

In [ ]:
import tpt_eng

s = tpt_eng.water_state(300.0, 3.0e6)
print(s.region, round(s.enthalpy_kj_per_kg, 3), round(s.entropy_kj_per_kg_k, 6))
print('saturation pressure at 100 °C:', round(tpt_eng.water_saturation_pressure_pa(373.15)), 'Pa')

Results that lie outside the validated range come back flagged, not rejected:

In [ ]:
t = tpt_eng.water_state_tagged(1100.0, 1.0e5)
print(t.in_range, t.warnings)
print(t.source)

## Moist air and dew point

In [ ]:
from tpt_eng import air

rh = air.relative_humidity(1000.0, 293.15)
print('relative humidity at 1000 Pa vapour, 20 °C:', round(rh, 4))
print('dew point (°C):', round(air.dew_point_k(1000.0) - 273.15, 2))
print('enthalpy (kJ/kg dry air):', round(air.moist_air_enthalpy_kj_per_kg(293.15, 0.01), 3))

## Fuels

In [ ]:
from tpt_eng.fuels import Fuel, BlendedFuel

for fuel in (Fuel.Methane, Fuel.Hydrogen, Fuel.Diesel):
    print(fuel, round(fuel.lhv_mj_kg(), 2), 'MJ/kg;', round(fuel.stoichiometric_air_fuel_ratio(), 2), 'kg air/kg fuel')
blend = BlendedFuel(0.2)
print('20 % H2 blend LHV:', round(blend.lhv_mj_kg, 2), 'MJ/kg')

## PID step response

In [ ]:
from tpt_eng.controls import Pid

pid = Pid(2.0, 1.0, 0.0, output_min=-5.0, output_max=5.0)
pid.set_setpoint(1.0)
y, dt = 0.0, 0.01
for _ in range(1000):
    u = pid.update(y, dt)
    y += dt * (u - y) / 1.0
print('output after 10 s:', round(y, 4))

## A simply supported beam

In [ ]:
from tpt_eng.structural import SimplySupportedBeam, section_utilization

beam = SimplySupportedBeam(4.0)
beam.add_uniform_load(0.0, 4.0, 8_000.0)  # 2 kN/m over 4 m
print('reaction A (N):', beam.reaction_a_n())
print('max moment (N·m):', round(beam.max_bending_moment_nm(resolution=400), 3))
print('utilisation for Z = 1e-4 m³, 200 MPa:', section_utilization(4000.0, 1.0e-4, 200.0e6))